# Домашняя работа 2 — качество данных

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
LOCAL = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
print("pandas", pd.__version__)

pandas 2.2.2


## Независимая загрузка

In [2]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
export = pd.read_csv(LOCAL / "training_export.csv", dtype="string")
raw_copy = export.copy(deep=True)
print(decisions.shape, export.shape)
print(export.dtypes)
export.head(5)

(6830, 7) (6832, 7)
source_row       string[python]
activity_year    string[python]
lei              string[python]
county_code      string[python]
action_taken     string[python]
loan_amount      string[python]
income           string[python]
dtype: object


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,BAD
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0


## H1. Собственная функция очистки

In [3]:
def clean_export(source):
    data = source.copy()
    original = source.copy()
    replay_mask = data.duplicated()
    replays = replay_mask.sum()
    data = data.loc[~replay_mask].copy()
    original = original.loc[~replay_mask].copy()
    known_rows = data["source_row"].notna()
    repeated_rows = data.loc[known_rows, "source_row"].duplicated(keep=False)
    conflict_ids = data.loc[known_rows, "source_row"][repeated_rows].unique()
    conflict_mask = data["source_row"].isin(conflict_ids)
    quarantine = original.loc[conflict_mask].copy()
    quarantine["reason"] = "conflicting source_row"
    data = data.loc[~conflict_mask].copy()
    original = original.loc[~conflict_mask].copy()
    data["county_code"] = data["county_code"].str.strip()
    data["lei"] = data["lei"].str.strip()
    parse_errors = 0
    for column in ["action_taken", "loan_amount", "income"]:
        if column in data.columns:
            old_values = data[column]
            new_values = pd.to_numeric(old_values, errors="coerce")
            errors = old_values.notna() & new_values.isna()
            parse_errors += errors.sum()
            data[column] = new_values
    good_lei = data["lei"].notna() & data["lei"].ne("")
    good_action = data["action_taken"].isin([1, 2, 3])
    good_amount = data["loan_amount"].notna() & (data["loan_amount"] > 0)
    good_county = data["county_code"].eq("11001")
    good_rows = good_lei & good_action & good_amount & good_county
    bad_rows = original.loc[~good_rows].copy()
    reasons = []
    for i in bad_rows.index:
        reason = []
        if not good_lei.loc[i]:
            reason.append("unknown lei")
        if not good_action.loc[i]:
            reason.append("wrong action_taken")
        if not good_amount.loc[i]:
            reason.append("wrong loan_amount")
        if not good_county.loc[i]:
            reason.append("wrong county_code")
        reasons.append(", ".join(reason))
    bad_rows["reason"] = reasons
    quarantine = pd.concat([quarantine, bad_rows])
    clean = data.loc[good_rows].copy()
    audit = {
        "input": len(source),
        "replays": int(replays),
        "clean": len(clean),
        "quarantine": len(quarantine),
        "parse_errors": int(parse_errors)
    }

    print("Журнал:", audit)
    print("Баланс:", audit["input"], "=", 
          audit["replays"] + audit["clean"] + audit["quarantine"])

    return clean, quarantine, audit

## H2. Год и известный доход

In [5]:
clean, quarantine, audit = clean_export(export)
print("\nПодготовленные данные:")
display(clean.head())
print("\nКарантин:")
display(quarantine.head())
clean, quarantine, audit = clean_export(export)
year_ok = clean["activity_year"].eq("2023").all()
print("Во всех подготовленных строках activity_year = '2023':", year_ok)
total = len(clean)
known_income = clean["income"].notna().sum()
unknown_income = clean["income"].isna().sum()
mean_income = clean["income"].mean()
print("Всего подготовленных строк:", total)
print("Известный income:", known_income)
print("Неизвестный income:", unknown_income)
print("Средний известный доход:", round(mean_income, 2))
print("Единица дохода: тысячи долларов США")
print("Баланс:", audit["input"], "=",
      audit["replays"] + audit["clean"] + audit["quarantine"])
assert_frame_equal(export, raw_copy)
print("Исходный export не изменился")

Журнал: {'input': 6832, 'replays': 2, 'clean': 6828, 'quarantine': 2, 'parse_errors': 2}
Баланс: 6832 = 6832

Подготовленные данные:


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income
2,5,2023,549300P1347BFCBHJV60,11001,1,275000.0,<NA>
3,7,2023,549300RPOGWJRH63HS39,11001,1,295000.0,63.0
4,8,2023,549300RPOGWJRH63HS39,11001,1,1125000.0,247.0
5,9,2023,549300RPOGWJRH63HS39,11001,1,305000.0,152.0
6,10,2023,549300RPOGWJRH63HS39,11001,1,505000.0,110.0



Карантин:


,source_row,activity_year,lei,county_code,action_taken,loan_amount,income,reason
0,2,2023,549300VZVN841I2ILS84,11001,1,oops,72.0,wrong loan_amount
1,3,2023,549300P1347BFCBHJV60,11001,88,585000.0,174.0,wrong action_taken


Журнал: {'input': 6832, 'replays': 2, 'clean': 6828, 'quarantine': 2, 'parse_errors': 2}
Баланс: 6832 = 6832
Во всех подготовленных строках activity_year = '2023': True
Всего подготовленных строк: 6828
Известный income: 6485
Неизвестный income: 343
Средний известный доход: 237.07
Единица дохода: тысячи долларов США
Баланс: 6832 = 6832
Исходный export не изменился


## H3. Проверка отчётного года

In [6]:
wrong_year = export.copy(deep=True)
wrong_year.loc[wrong_year.index[0], "activity_year"] = "2022"
year_ok = wrong_year["activity_year"].eq("2023").fillna(False).all()
print("Проверка года после изменения:", year_ok)
print("Нарушение обнаружено:", not year_ok)
assert_frame_equal(export, raw_copy)
print("Исходный export сохранился без изменений")

Проверка года после изменения: False
Нарушение обнаружено: True
Исходный export сохранился без изменений


## H+1. Конфликт исходных значений

## H+2. Совпадение опубликованных полей